# Task 4 — FICO Score Quantization & Rating Map

JPMorgan Chase Quantitative Research — Forage Simulation

Create a **rating map** that converts borrower FICO scores into credit ratings using **quantization**. A **lower rating means better credit** (rating 1 = highest FICO bucket).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import pandas as pd

from fico_quantization import (
    DEFAULT_NUM_BUCKETS,
    bucket_summary,
    generate_log_likelihood_buckets,
    generate_mse_buckets,
    load_data,
    map_rating,
)

plt.style.use("seaborn-v0_8-whitegrid")
NUM_BUCKETS = DEFAULT_NUM_BUCKETS
print("Setup complete.")

## 1. Load Data & Explore FICO Scores

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from fico_quantization import load_data

df = load_data()

print(f"Borrowers:      {len(df):,}")
print(f"FICO range:     {df['fico_score'].min()} – {df['fico_score'].max()}")
print(f"Unique scores:  {df['fico_score'].nunique()}")
print(f"Default rate:   {df['default'].mean():.1%}")
df[['fico_score', 'default']].head()

## 2. Why Quantization?

FICO scores are continuous, but risk models often need **discrete rating labels**. Quantization finds bucket boundaries that best summarize the data.

We compare two objectives:
- **Mean Squared Error (MSE)** — minimize approximation error within each bucket
- **Log-Likelihood** — maximize fit to default/non-default distribution per bucket

Both are solved with **dynamic programming** over contiguous FICO buckets.

## 3. MSE-Optimal Buckets

Minimize within-bucket squared error:

$$\text{MSE} = \sum_{i=1}^{k} \sum_{x \in B_i} (x - \bar{x}_i)^2$$

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from fico_quantization import generate_mse_buckets, bucket_summary, load_data

df = load_data()
mse_buckets = generate_mse_buckets(NUM_BUCKETS, df)

print(f"MSE-optimal boundaries ({NUM_BUCKETS} buckets):")
print(mse_buckets.boundaries)
print(f"\nTotal MSE: {mse_buckets.mse:,.2f}")

mse_summary = bucket_summary(df, mse_buckets)
mse_summary

## 4. Log-Likelihood-Optimal Buckets

Maximize log-likelihood using default counts per bucket:

$$\mathcal{L} = \sum_i \left[ k_i \log p_i + (n_i - k_i) \log(1 - p_i) \right]$$

where $n_i$ = records in bucket $i$, $k_i$ = defaults, $p_i = k_i / n_i$.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from fico_quantization import generate_log_likelihood_buckets, bucket_summary, load_data

df = load_data()
ll_buckets = generate_log_likelihood_buckets(NUM_BUCKETS, df)

print(f"Log-likelihood boundaries ({NUM_BUCKETS} buckets):")
print(ll_buckets.boundaries)
print(f"\nTotal log-likelihood: {ll_buckets.log_likelihood:,.2f}")

ll_summary = bucket_summary(df, ll_buckets)
ll_summary

## 5. Compare Methods

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
from fico_quantization import (
    generate_log_likelihood_buckets,
    generate_mse_buckets,
    bucket_summary,
    load_data,
)

df = load_data()
mse_buckets = generate_mse_buckets(NUM_BUCKETS, df)
ll_buckets = generate_log_likelihood_buckets(NUM_BUCKETS, df)
mse_summary = bucket_summary(df, mse_buckets)
ll_summary = bucket_summary(df, ll_buckets)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(mse_summary["rating"], mse_summary["default_rate"], color="#2e75b6")
axes[0].set_title("MSE Buckets — Default Rate by Rating", fontweight="bold")
axes[0].set_xlabel("Rating (1 = best)")
axes[0].set_ylabel("Default Rate")
axes[0].invert_xaxis()

axes[1].bar(ll_summary["rating"], ll_summary["default_rate"], color="#c55a11")
axes[1].set_title("Log-Likelihood Buckets — Default Rate by Rating", fontweight="bold")
axes[1].set_xlabel("Rating (1 = best)")
axes[1].set_ylabel("Default Rate")
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

## 6. Rating Map Function

This is the deliverable: map a FICO score to a rating (1 = best credit).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from fico_quantization import map_rating


def get_credit_rating(fico_score: int, num_buckets: int = NUM_BUCKETS, method: str = "log_likelihood") -> int:
    """
    Map a FICO score to a credit rating.

    Parameters
    ----------
    fico_score : int
        Borrower FICO score.
    num_buckets : int
        Number of rating buckets.
    method : str
        'mse' or 'log_likelihood' boundary optimization.

    Returns
    -------
    int
        Rating from 1 (best) to num_buckets (worst).
    """
    return map_rating(fico_score, num_buckets=num_buckets, method=method)